## 바이오 빅데이터 분석 (1A): Python/Pandas Bridge + Visualization Practice

### 이 노트북의 목적
이 노트북은 파이썬 기초 문법을 한 번 배운 수강생이 실제 바이오 빅데이터 분석 코드로 넘어가기 위한 실습 자료입니다.  
문법 자체를 길게 반복하기보다는, 생명과학/의학 데이터 분석에서 자주 쓰는 `pandas DataFrame`과 기본 시각화 흐름을 실제 데이터로 연습합니다.

### 실습 주제
1. 바이오 데이터 테이블 구조 이해와 pandas DataFrame 핵심 복습
2. TCGA-BRCA 유방암 데이터의 gene expression과 clinical metadata 다루기
3. DataFrame 선택, 조건 필터링, `groupby`, table 결합 연습
4. 데이터 분포 시각화: histogram, pairplot, boxplot, violin plot, heatmap

### 사용할 데이터
1. CancerSEEK data: 단백질 marker와 임상 정보
2. TCGA-BRCA cohort: 유방암 gene expression과 clinical subtype
3. CCLE-CTRPv2: 암세포주 gene expression과 약물 반응 AUC

#### Machine Learning and Bioinformatics (MLBI) Lab


## 0. 필요한 패키지 설치 (처음 한번만 하면 됨)

In [ ]:
# 수업 환경에서 mlbi-lab 또는 scikit-network가 없을 때만 주석을 해제하고 실행하세요.
# !pip install mlbi-lab scikit-network statannotations

In [ ]:
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pearsonr
from scipy.stats import ttest_ind, mannwhitneyu, f_oneway, kruskal

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn import cluster, mixture
from sklearn.neighbors import kneighbors_graph

from sknetwork.clustering import Louvain
from mlbi.datasets import load_data

## 1. Data Frame의 구조와 활용법 실습

### 분석 관점에서 보는 DataFrame

생명과학/의학 데이터 분석에서는 대부분의 자료가 표 형태로 정리됩니다.

- 행(row): 샘플, 환자, 세포주, 실험 조건
- 열(column): 유전자, 단백질, 임상 변수, 약물 반응값
- index: 각 행을 구분하는 이름 또는 ID
- values: 실제 측정값

엑셀과 비슷해 보이지만, `pandas DataFrame`은 행/열 선택, 조건 필터링, 통계 요약, 시각화 연동을 코드로 재현할 수 있다는 점이 중요합니다.

In [ ]:
# 아주 작은 예제 DataFrame을 먼저 만들어 봅니다.
# 실제 바이오 데이터도 구조는 이 예제와 같습니다.

example_df = pd.DataFrame({
    'sample_id': ['S1', 'S2', 'S3', 'S4'],
    'subtype': ['Lum.A', 'Lum.B', 'Basal', 'Her2'],
    'ESR1': [12.4, 8.1, 1.2, 4.5],
    'ERBB2': [3.1, 4.0, 2.8, 14.2],
    'MKI67': [2.3, 7.8, 9.1, 6.5]
})

example_df

### 빠른 복습: 한 열과 여러 열 선택

`df['ESR1']`은 한 열을 `Series`로 가져오고, `df[['ESR1', 'ERBB2']]`는 여러 열을 `DataFrame`으로 가져옵니다.  
대괄호가 하나인지 두 개인지에 따라 결과의 타입이 달라진다는 점을 자주 확인하세요.

In [ ]:
print(type(example_df['ESR1']))
print(type(example_df[['ESR1', 'ERBB2']]))

example_df[['ESR1', 'ERBB2']]

### 빠른 복습: 조건으로 샘플 고르기

아래 예시는 `ESR1` 발현량이 높은 샘플만 고르는 코드입니다.  
실제 분석에서는 특정 subtype, 특정 stage, 특정 발현량 기준에 맞는 샘플을 고를 때 같은 방식을 씁니다.

In [ ]:
high_esr1 = example_df[example_df['ESR1'] > 5]
high_esr1

### 데이터 불러오기 - CancerSEEK data

In [ ]:
load_data()

In [ ]:
data = load_data('cancerseek')
data.keys()

In [ ]:
df_pep = data['protein_expression']
df_clinical = data['clinical_info']

df_pep.head()

### 다른 데이터셋 - TCGA BRCA gene expression and clinical data

In [ ]:
data = load_data('tcga-brca')
data.keys()

In [ ]:
df_gep = data['gene_expression']
df_clinical = data['clinical_info']

In [ ]:
df_clinical.head()

In [ ]:
df_clinical.columns.values

In [ ]:
df_gep.shape, df_clinical.shape

In [ ]:
df_gep

In [ ]:
df_gep.sum(axis = 1)

### DataFrame 활용 연습
1. head() method를 이용한 Data frame 확인: head(), tail()
2. shape attribute를 이용한 size 확인: shape
3. row name(index) 및 column name 확인
4. index/column name을 이용한 행/열 가져오기: loc(), iloc()
5. pandas series
6. NA(None)이 포함된 element 확인하기: isnull()
7. 행/열의 합/평균/표준편차 구하기: sum(), mean(), std()
8. 행/열의 최대값/최소값/최대값의 index 구하기: max(), min(), idxmax(), idxmin()
9. 범주형 데이터의 범주 카운트 구하기: value_counts()
10. 조건에 맞는 행들만 선택하기 - boolean indexing
11. 범주형 데이터가 특정 집합의 원소인지 아닌지 조사하기: isin()
12. 등호/부등호의 사용: ==, >, <
13. 특정 열 기준으로 정렬하기: sort_values()

In [ ]:
# TCGA-BRCA gene expression table과 clinical table을 이용해 DataFrame 기본기를 연습합니다.

print('gene expression shape:', df_gep.shape)
print('clinical info shape:', df_clinical.shape)

# 1) 앞/뒤 일부 행 확인
print('--- df_gep head ---')
display(df_gep.head())

print('--- df_clinical head ---')
display(df_clinical.head())

### DataFrame 핵심 조작 예제

아래 셀들은 실제 분석에서 거의 매번 쓰는 조작입니다.  
처음에는 코드를 외우기보다, “행을 고르는지, 열을 고르는지, 조건을 거는지”를 구분해서 읽는 것이 중요합니다.

In [ ]:
# 열 이름 확인
print(df_gep.columns[:10])
print(df_clinical.columns)

# 관심 유전자 3개만 선택
marker_genes = ['ESR1', 'ERBB2', 'MKI67']
df_marker = df_gep[marker_genes]

df_marker.head()

In [ ]:
# loc: index/column 이름으로 선택
# iloc: 행/열 번호 위치로 선택

print('첫 번째 샘플 ID:', df_gep.index[0])

display(df_gep.loc[df_gep.index[0], marker_genes])
display(df_gep.iloc[:5, :5])

In [ ]:
# 임상 정보에서 subtype별 샘플 수 확인
# value_counts()는 범주형 변수의 분포를 볼 때 자주 사용합니다.

df_clinical['subtype'].value_counts(dropna=False)

In [ ]:
# gene expression과 clinical subtype을 하나의 DataFrame으로 합치기
# 두 table의 index가 같은 sample ID라고 가정하고 열 방향(axis=1)으로 붙입니다.

practice_df = pd.concat([
    df_gep[marker_genes],
    df_clinical[['subtype']]
], axis=1)

practice_df.head()

In [ ]:
# 조건 필터링: Basal-like 샘플만 선택
basal_df = practice_df[practice_df['subtype'] == 'Basal']

print(basal_df.shape)
basal_df.head()

In [ ]:
# groupby: subtype별 marker gene 평균과 표준편차 계산
summary_mean = practice_df.groupby('subtype')[marker_genes].mean()
summary_std = practice_df.groupby('subtype')[marker_genes].std()

print('Subtype별 평균')
display(summary_mean)

print('Subtype별 표준편차')
display(summary_std)

### 다른 데이터셋 - CCLE cellline gene expression and drug response measurement

In [ ]:
data = load_data('ccle-ctrpv2')
data.keys()

In [ ]:
df_gexp = data['gene_expression']
df_cl_info = data['cellline_info']
df_auc = data['auc']
df_ec50 = data['ec50']
df_drug_info = data['drug_info']

### Homework

유방암은 크게 3가지의 subtype (Basal-like, Luminal 및 Her2)으로 구분한다. 또한, Lum.subtype은 다시 liminal A와 Luminal B로 나뉜다. 이들 subtype들은 대체로 3가지 유전자(ESR1, ERBB2, MKI67)의 발현량에 따라 구분되기도 한다. 3가지의 subtype (Basal-like, Luminal 및 Her2) 각각에 대해 3가지 유전자 (ESR1, ERBB2, MKI67)의 발현량의 평균과 표준편자를 각각 구하여 data frame으로 요약해 보아라. 이 때 data frame의 index는 subtype의 이름, columns는 유전자 이름으로 설정하라. (예, 'ESR1 (mean)', 'ESR1 (stdev)' 등) 이를 위해 필요한 정보는 TCGA-BRCA 데이터에 모두 포함되어 있으며 해당 데이터를 잘 살펴보고 필요 시 다음의 인터넷 사이트를 참고하라. NA가 포함된 부분은 적절히 제외하라.
https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html    

#### Homework 접근 힌트

1. `df_gep[['ESR1', 'ERBB2', 'MKI67']]`로 관심 유전자만 고릅니다.
2. `df_clinical[['subtype']]`을 함께 붙여 분석용 DataFrame을 만듭니다.
3. `dropna()`로 subtype 또는 발현량 결측치를 정리합니다.
4. `groupby('subtype')` 뒤에 `mean()`과 `std()`를 적용합니다.
5. 결과를 보기 좋은 하나의 DataFrame으로 정리합니다.

In [ ]:
# Homework template
# 아래 빈칸을 채워 subtype별 ESR1/ERBB2/MKI67 평균과 표준편차를 계산해 보세요.

marker_genes = ['ESR1', 'ERBB2', 'MKI67']

hw_df = pd.concat([
    df_gep[marker_genes],
    df_clinical[['subtype']]
], axis=1)

hw_df = hw_df.dropna(subset=marker_genes + ['subtype'])

# subtype_summary = ...
# subtype_summary

## 2. 데이터 분포의 시각화 연습

시각화는 “분석 결과를 예쁘게 보여주는 단계”라기보다, 데이터가 어떤 모양인지 먼저 확인하는 탐색 과정입니다.  
특히 바이오 데이터에서는 극단값, subtype별 차이, marker gene 간 관계를 눈으로 확인하는 과정이 중요합니다.

1. Histogram: 한 변수의 전체 분포
2. Pairplot: 여러 변수 간 관계
3. Box plot: 그룹별 분포 비교
4. Violin plot: 그룹별 분포 모양 비교
5. Heatmap / clustermap: 여러 유전자 패턴 비교

In [ ]:
data = load_data('tcga-brca')

df_gep = data['gene_expression']
df_clinical = data['clinical_info']

In [ ]:
df_log_gep = np.log10(df_gep + 1)

In [ ]:
df_log_gep['ESR1'].hist(bins = 40, alpha = 0.5)
df_log_gep['ERBB2'].hist(bins = 40, alpha = 0.5)

plt.xlabel('log(TPM)', fontsize = 12)
plt.ylabel('N samples', fontsize = 12)
plt.legend(['ESR1', 'ERBB2'])

#### pair wise scatter plot 그리기

In [ ]:
df = df_log_gep[['ESR1', 'ERBB2', 'MKI67']].copy(deep = True)
df['subtype'] = df_clinical['subtype'].copy(deep=True)
df

In [ ]:
df['subtype'].value_counts()

In [ ]:
df.head()

In [ ]:
df['subtype'].value_counts()

#### Pairplot 해석 포인트

각 점은 한 환자 샘플이고, 색은 유방암 subtype입니다.  
`ESR1`, `ERBB2`, `MKI67` 세 유전자만으로도 일부 subtype이 서로 다른 위치에 모이는지 관찰해 봅니다.

In [ ]:
## Pairplot with hue
sns.pairplot(df, hue = 'subtype', diag_kind = 'kde')

#### Box plot 그리기

In [ ]:
sns.boxplot(data = df.iloc[:,:3])

In [ ]:
sns.boxplot(data = df, y = 'ERBB2', x = 'subtype', hue = 'subtype')

#### Violin plot 그리기

In [ ]:
sns.violinplot(data = df, y = 'ERBB2', x = 'subtype', hue = 'subtype')

#### Heatmap 그리기

In [ ]:
PAM50_genes = ['ACTR3B', 'ANLN', 'BAG1', 'BCL2', 'BIRC5', 'BLVRA',
 'CCNB1', 'CCNE1', 'CDC20', 'CDC6', 'CDH3', 'CENPF', 'CEP55',
 'CXXC5', 'EGFR', 'ERBB2', 'ESR1', 'EXO1', 'FGFR4', 'FOXA1',
 'FOXC1', 'GPR160', 'GRB7', 'KIF2C', 'KRT14', 'KRT17', 'KRT5',
 'MAPT', 'MDM2', 'MELK', 'MIA', 'MKI67', 'MLPH', 'MMP11', 'MYBL2',
 'MYC', 'NAT1', 'NDC80', 'NUF2', 'ORC6L', 'PGR', 'PHGDH', 'PTTG1',
 'RRM2', 'SFRP1', 'SLC39A6', 'TMEM45B', 'TYMS', 'UBE2C', 'UBE2T']

PAM50_genes = list(set(PAM50_genes).intersection(df_log_gep.columns.values))
len(PAM50_genes)

In [ ]:
X_sel = df_log_gep[PAM50_genes] # .copy(deep=True)
# X_sel = df_log_gep[['ESR1', 'ERBB2', 'MKI67']]
y = df_clinical['subtype']

plt.figure(num=None, figsize=(12, 5), dpi=80, facecolor='w', edgecolor='k')
# ax = sb.heatmap(X_sel.transpose())

lut = dict(zip(set(y), "rbgym"))
col_colors = y.map(lut)
dfc = pd.DataFrame([col_colors]).transpose()

g = sns.clustermap(X_sel.transpose(), col_cluster = True, row_cluster = True, standard_scale = 0,
                   col_colors = dfc, cmap="mako", figsize = (12,8))
g.fig.suptitle('Heatmap\n ', fontsize=20, va = 'center')

# Draw the legend bar for the classes
for label in list(set(y)):
    g.ax_col_dendrogram.bar(0, 0, color=lut[label],
                            label=label, linewidth=0)
g.ax_col_dendrogram.legend(loc="center right", bbox_to_anchor=(1.25, 0.5), ncol=1)
# ax.cax.set_position([.2, .2, .03, 1])
g.ax_heatmap.set_xlabel('Sample')
g.ax_heatmap.set_ylabel('Gene')

### Homework

유방암은 크게 3가지의 subtype (Basal-like, Luminal 및 HER2-enriched)으로 구분한다. 또한, Luminal subtype은 다시 liminal A와 Luminal B로 나뉜다. 이들 subtype들은 대체로 3가지 유전자(ESR1, ERBB2, MKI67)의 발현량에 따라 구분되기도 한다.

1. 3가지의 subtype (Basal-like, Luminal 및 HER2-enriched) 각각에 대해 3가지 유전자 (ESR1, ERBB2, MKI67)의 발현량를 Violin plot으로 그려서 비교해 보아라. NA가 포함된 부분은 적절히 제외하라.
2. 3가지의 subtype (Basal-like, Luminal 및 HER2-enriched) 각각에 대해 3가지 유전자 (ESR1, ERBB2, MKI67)의 발현량를 Cluster Map으로 그려서 비교해 보아라.
3. 1/2의 비교 굘과에 대한 간략한 고찰을 제시하여라.